# SimPro-ValueNetwork — Comprehensive Colab Validation

Use a fresh Colab runtime and select **Run all**. All 60 released models are checked by default: real shell commands using `!` run first, followed by ordinary Python API cells. Checks also cover installation, data preparation, training, fine-tuning, resume, checkpoint saving, and original-file preservation. The package/import name is `simpro_vn`; console commands are `simpro-vn` and `simpro-vn-train`.

A GPU runtime is recommended. CPU execution is supported but the model matrix and production-network fine-tuning can be slow. Unavailable CUDA/bf16 checks are explicitly reported as SKIP, giving PASS_WITH_SKIPS rather than claiming GPU coverage. This validates behavior and consistency, not research accuracy or full-dataset training. Full-cache research figure reproduction is documented separately.

This notebook has no precomputed results. Each execution creates a new RUN_DIR for reports and checkpoints; released weights are read and checked by hash.


## Paths and Run Directory

Upload/extract the full project and set PROJECT_DIR to the folder containing pyproject.toml. Weights may live elsewhere: set WEIGHTS_DIR and retain the smiles, smarts, safe, and selfies subdirectories. Drive is optional; supply your own path. This notebook does not download weights.


In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os
import sys
import time
import traceback
import hashlib

USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_DIR = Path('/content/SimPro-ValueNetwork').expanduser().resolve()
WEIGHTS_DIR = PROJECT_DIR / 'weights'
RUN_DIR = Path('/content/validation_runs') / ('comprehensive_' + uuid4().hex)
DEVICE = 'auto'  # auto, cpu, or cuda
assert (PROJECT_DIR / 'pyproject.toml').is_file(), PROJECT_DIR
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONIOENCODING'] = 'utf-8'
phases = []

def save_summary():
    status = 'FAIL' if any(p['status'] == 'FAIL' for p in phases) else 'IN_PROGRESS'
    (RUN_DIR / 'summary.json').write_text(json.dumps(
        {'project': 'SimPro-ValueNetwork', 'status': status, 'phases': phases},
        ensure_ascii=False, indent=2), encoding='utf-8')

def phase(name, action):
    started = time.perf_counter()
    row = {'phase': name}
    try:
        value = action()
        row.update(status='PASS', detail=value)
    except Exception:
        row.update(status='FAIL', detail=traceback.format_exc())
        raise
    finally:
        row['seconds'] = round(time.perf_counter() - started, 3)
        phases.append(row)
        save_summary()
        print(name, row['status'], flush=True)
    return value

def shell_finished(name):
    exit_file = RUN_DIR / (name + '.exitcode')
    def check():
        assert exit_file.is_file(), 'Command was not executed'
        status = int(exit_file.read_text().strip())
        assert status == 0, f'{name}: exit {status}; See the output above'
        return {'returncode': status}
    return phase(name, check)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def load_helper(name, path):
    import importlib.util
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

print('Project:', PROJECT_DIR)
print('Weights:', WEIGHTS_DIR)
print('Reports and new checkpoints:', RUN_DIR)


## Normal Installation and Console Entrypoints

Perform a normal installation including data-preparation dependencies, without editable installation or sys.path shortcuts. Restart the runtime first if an older package was already imported. `!` is Colab/IPython shell syntax; Python then checks the exit code. Command output alone does not establish success.


In [ ]:
!"{sys.executable}" -m pip install "{PROJECT_DIR}[data]"; echo $? > "{RUN_DIR}/install.exitcode"
shell_finished('install')
!simpro-vn --help; echo $? > "{RUN_DIR}/cli_help.exitcode"
shell_finished('cli_help')
!simpro-vn-train --help; echo $? > "{RUN_DIR}/training_help.exitcode"
shell_finished('training_help')
!"{sys.executable}" -m simpro_vn --help; echo $? > "{RUN_DIR}/module_help.exitcode"
shell_finished('module_help')


## Installed Location, Environment, and File Integrity

Verify 60 released checkpoints, 40 tokenizer/mapping files, model descriptions, and installed copies. WEIGHTS_DIR is configurable; retain catalog filenames and representation subdirectories. Snapshot all .pt hashes for the final preservation check.


In [ ]:
import importlib.metadata
import torch
import simpro_vn
from simpro_vn import ValueNetworkPredictor, identify_checkpoint
from simpro_vn.bundles import ASSETS
from simpro_vn.scaling import PROPERTY_NAMES

def inventory():
    module = Path(simpro_vn.__file__).resolve()
    assert (PROJECT_DIR / 'src').resolve() not in module.parents, 'Restart and use normal installation'
    manifest = json.loads((PROJECT_DIR / 'weights/manifest.json').read_text())['models']
    assert len(manifest) == 60
    for item in manifest:
        path = WEIGHTS_DIR / Path(item['path']).relative_to('weights')
        assert path.is_file(), path
        assert path.stat().st_size == item['bytes'] and sha256(path) == item['sha256'], path
    tokens = json.loads((PROJECT_DIR / 'docs/tokenizer_manifest.json').read_text())['files']
    assert len(tokens) == 40
    for item in tokens:
        source = PROJECT_DIR / item['path']
        installed = ASSETS / Path(item['path']).relative_to('src/simpro_vn/assets')
        assert sha256(source) == sha256(installed) == item['sha256'], source
    catalog = json.loads((ASSETS / 'pretrained_models.json').read_text())
    assert len(catalog['models']) == 60
    for path in (PROJECT_DIR / 'weights/model_descriptions.json', PROJECT_DIR / 'src/simpro_vn/assets/pretrained_models.json'):
        assert catalog == json.loads(path.read_text()), path
    assert sha256(ASSETS / 'value_network_scaler.json') == sha256(PROJECT_DIR / 'src/simpro_vn/assets/value_network_scaler.json')
    return {'models': 60, 'tokenizer_files': 40, 'installed_module': str(module),
            'version': importlib.metadata.version('simpro_vn'), 'torch': str(torch.__version__),
            'cuda_available': torch.cuda.is_available(), 'requested_device': DEVICE}

environment = phase('inventory', inventory)
original_weights = {str(p.resolve()): sha256(p) for p in WEIGHTS_DIR.rglob('*.pt')}
assert len(original_weights) >= 60
(RUN_DIR / 'original_weight_hashes.json').write_text(json.dumps(original_weights, indent=2))
print(json.dumps(environment, indent=2))


## Wheel and Functional Tests (! Commands)

Wheel validation checks imports, assets, API, and CLI in a temporary venv outside the source tree while reusing current dependencies such as PyTorch. Functional tests cover loss, causality, tensors, invalid inputs, representations, curricula, training/fine-tuning/resume, reload, and output protection. Unavailable CUDA produces explicit skips. These tests use tiny models and synthetic labels; they do not assess scientific accuracy.


In [ ]:
!"{sys.executable}" "{PROJECT_DIR}/examples/validate_installation.py" --project-dir "{PROJECT_DIR}" --output-dir "{RUN_DIR}/installation"; echo $? > "{RUN_DIR}/wheel.exitcode"
shell_finished('wheel')
installation_report = json.loads((RUN_DIR / 'installation/report.json').read_text())
assert installation_report['status'] == 'PASS'

!"{sys.executable}" "{PROJECT_DIR}/examples/run_validation_tests.py" --project-dir "{PROJECT_DIR}" --output-dir "{RUN_DIR}/functional_tests"; echo $? > "{RUN_DIR}/functional_tests.exitcode"
shell_finished('functional_tests')
test_report = json.loads((RUN_DIR / 'functional_tests/report.json').read_text())
assert test_report['tests_run'] > 0 and not test_report['failures'] and not test_report['errors']
print('Skipped capabilities:', test_report['skipped'])


## All 60 Released Models: CLI First

These runnable `!simpro-vn` examples cover complete strings, prefixes, trajectories, and file input. The following shell command validates CLI behavior across all 4 × 5 × 3 models, including invalid arguments, for 182 checks. Runtime depends on CPU/GPU and storage speed.


In [ ]:
SOURCE_CHECKPOINT = WEIGHTS_DIR / 'smiles/value_network_best_smiles_bpe_min_freq_5000_d6.pt'
(RUN_DIR / 'molecules.txt').write_text('CCO\nCCN\n', encoding='utf-8')
!simpro-vn --checkpoint "{SOURCE_CHECKPOINT}" --depth 6 --device "{DEVICE}" --text CCO --output "{RUN_DIR}/direct_complete.json"; echo $? > "{RUN_DIR}/direct_complete.exitcode"
shell_finished('direct_complete')
!simpro-vn --checkpoint "{SOURCE_CHECKPOINT}" --depth 6 --device "{DEVICE}" --text CC --prefix --output "{RUN_DIR}/direct_prefix.json"; echo $? > "{RUN_DIR}/direct_prefix.exitcode"
shell_finished('direct_prefix')
!simpro-vn --checkpoint "{SOURCE_CHECKPOINT}" --depth 6 --device "{DEVICE}" --text CCO --trajectory --output "{RUN_DIR}/direct_trajectory.json"; echo $? > "{RUN_DIR}/direct_trajectory.exitcode"
shell_finished('direct_trajectory')
!simpro-vn --checkpoint "{SOURCE_CHECKPOINT}" --depth 6 --device "{DEVICE}" --input-file "{RUN_DIR}/molecules.txt" --batch-size 2 --output "{RUN_DIR}/direct_batch.json"; echo $? > "{RUN_DIR}/direct_batch.exitcode"
shell_finished('direct_batch')

MATRIX_DIR = RUN_DIR / 'published_matrix'
!"{sys.executable}" "{PROJECT_DIR}/examples/colab_validation.py" --project-dir "{PROJECT_DIR}" --weights-dir "{WEIGHTS_DIR}" --output-dir "{MATRIX_DIR}" --device "{DEVICE}" --phase cli; echo $? > "{RUN_DIR}/published_cli.exitcode"
shell_finished('published_cli')


## Python Cells: All 60 APIs and Numerical Agreement

Continue from CLI results to check SHA256 descriptions, relative/custom configurations, batching, tensors, IDs, prefixes, trajectories, KV-cache, EOS, reset, and invalid inputs. Expect 360 API checks and 542 combined CLI/API checks. Models load sequentially and are released between cases.


In [ ]:
validation_helpers = load_helper('simpro_validation_helpers', PROJECT_DIR / 'examples/colab_validation.py')

def published_api():
    runner = validation_helpers.ColabValidation(
        PROJECT_DIR, WEIGHTS_DIR, MATRIX_DIR,
        representations=('smiles', 'smarts', 'safe', 'selfies'),
        settings=(5000, 10000, 20000, 50000, 'no_bpe'), depths=(2, 4, 6), device=DEVICE)
    report = json.loads((MATRIX_DIR / 'report.json').read_text())
    assert len(report['records']) == 182 and all(row['status'] == 'PASS' for row in report['records'])
    runner.records = report['records']
    runner.cli_results = json.loads((MATRIX_DIR / 'cli_predictions.json').read_text())
    runner.preflight()
    runner.run_api()
    runner.summary()
    assert len(runner.records) == 542
    return {'models': 60, 'cli_checks': 182, 'api_checks': 360, 'report': str(MATRIX_DIR / 'report.json')}

phase('published_api', published_api)


## Integration Interfaces and CLI Agreement

The following uses ordinary Python syntax suitable for integration. Tensors default to property units; standardized=True returns z-scores. Outputs do not track Value Network gradients and can act as fixed scores for another model's training; this does not make discrete token IDs differentiable.


In [ ]:
def integration_api():
    predictor = ValueNetworkPredictor(checkpoint_path=SOURCE_CHECKPOINT, depth=6, device=DEVICE)
    print(json.dumps(predictor.describe(), indent=2))
    assert identify_checkpoint(SOURCE_CHECKPOINT)['depth'] == 6
    texts = ['CCO', 'CCN']
    values = predictor.predict_tensor(texts)
    zscores = predictor.predict_tensor(texts, standardized=True)
    torch.testing.assert_close(predictor.scaler.inverse_transform(zscores), values)
    expected = torch.tensor([[row[name] for name in PROPERTY_NAMES]
                             for row in json.loads((RUN_DIR / 'direct_batch.json').read_text())], device=values.device)
    expected_z = (expected - predictor.scaler.mean.to(values.device)) / predictor.scaler.std.to(values.device)
    torch.testing.assert_close(zscores, expected_z, rtol=1e-4, atol=1e-4)
    sequences = [torch.tensor(predictor.encode(text)) for text in texts]
    ids = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=predictor.tokenizer.pad_id)
    mask = ids.ne(predictor.tokenizer.pad_id)
    torch.testing.assert_close(predictor.predict_ids(ids, attention_mask=mask), values)
    all_tokens = predictor.predict_ids(ids, attention_mask=mask, output_all_tokens=True)
    assert torch.all(all_tokens[~mask.to(all_tokens.device)] == 0)
    session = predictor.new_session()
    pieces = [session.step_tensor(sequences[0][:2]), session.step_tensor(sequences[0][2:])]
    torch.testing.assert_close(torch.cat(pieces), all_tokens[0, :len(sequences[0])])
    session.reset()
    assert session.length == 0 and session.caches is None
    scores = predictor.predict_tensor(texts, standardized=True)
    assert not scores.requires_grad and not torch.is_inference(scores)
    generator_parameter = torch.ones_like(scores, requires_grad=True)
    (generator_parameter * scores).sum().backward()
    torch.testing.assert_close(generator_parameter.grad, scores)
    assert all(parameter.grad is None for parameter in predictor.model.parameters())
    result = {'shape': list(values.shape), 'device': str(values.device), 'verification': predictor.describe()['verification']}
    if torch.cuda.is_available() and predictor.device.type == 'cuda':
        cpu = ValueNetworkPredictor(checkpoint_path=SOURCE_CHECKPOINT, depth=6, device='cpu')
        torch.testing.assert_close(cpu.predict_tensor(texts, standardized=True), zscores.cpu(), rtol=1e-4, atol=1e-4)
        result['cpu_cuda_parity'] = 'PASS'
        del cpu
    else:
        result['cpu_cuda_parity'] = 'SKIP: CUDA was not selected or is unavailable'
    return result

integration_report = phase('integration_api', integration_api)
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Small Data Preparation: Stereo, 3D Optimization, 17 Properties

Process a few simple molecules and one with unspecified stereochemistry through the complete preparation example. Every accepted structure undergoes 3D optimization and property calculation, followed by fixed tokenizer/mapping/scaler compatibility and parent-group splitting. Save and report each stage's rejected rows; acceptance is not assumed. This small dataset only checks the workflow.


In [ ]:
def prepare_example():
    import pandas as pd
    import numpy as np
    helper = load_helper('simpro_data_preparation', PROJECT_DIR / 'examples/finetuning_data.py')
    destination = RUN_DIR / 'data_preparation'
    destination.mkdir(exist_ok=False)
    data = pd.DataFrame({'SMILES': ['CCO', 'CCN', 'CCC', 'CC(O)C(=O)O']})
    expanded, rejected1 = helper.expand_smiles(data, enumerate_stereo=True)
    assert len(expanded) >= len(data)
    optimized, rejected2 = helper.optimize_all(expanded)
    assert not optimized.empty
    assert set(optimized['Optimization_Status']) == {'Converged'}
    labeled, rejected3 = helper.calculate_all(optimized)
    predictor = ValueNetworkPredictor(checkpoint_path=SOURCE_CHECKPOINT, depth=6, device='cpu')
    ready, rejected4 = helper.check_compatibility(labeled, predictor)
    assert len(ready) >= 2, 'Too few accepted molecules; inspect rejected CSVs'
    assert np.isfinite(ready[list(PROPERTY_NAMES)].to_numpy(dtype=float)).all()
    split = helper.split_by_parent(ready, val_fraction=0.25)
    assert set(split['Split_Set']) == {'Train', 'Val'}
    assert split.groupby('Parent_Group')['Split_Set'].nunique().max() == 1
    split.to_csv(destination / 'prepared.csv', index=False)
    rejections = []
    for name, rejected in zip(('enumeration', 'optimization', 'properties', 'compatibility'),
                              (rejected1, rejected2, rejected3, rejected4)):
        rejected.to_csv(destination / (name + '_rejected.csv'), index=False)
        rejections.append({'stage': name, 'rejected': len(rejected)})
    result = {'input_rows': len(data), 'expanded_rows': len(expanded), 'accepted_rows': len(split), 'rejections': rejections}
    (destination / 'report.json').write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2))
    return result

phase('data_preparation', prepare_example)


## Direct ! Training, Then Python Fine-Tuning and Resume

Select two Train and two Val records from the bundled 300-row real-label CSV; Test data are not used for model selection. Demonstrate scratch training with `!simpro-vn-train` on a tiny model. Then use Python to fine-tune the released SMILES 5000 d6 model for one epoch and resume epoch two from last.pt. Every phase writes a new subdirectory and verifies best.pt, last.pt, history, and prediction reload. Production-model fine-tuning uses fp32 and batch size 2; this is not a full performance evaluation.


In [ ]:
import csv
from dataclasses import asdict, replace
from simpro_vn.training import TrainingConfig, run_training

with (PROJECT_DIR / 'examples/data/value_network_smoke.csv').open(encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    fields = reader.fieldnames
    all_rows = list(reader)
selected = [row for split in ('Train', 'Val') for row in [r for r in all_rows if r['Split_Set'] == split][:2]]
assert len(selected) == 4 and {r['Split_Set'] for r in selected} == {'Train', 'Val'}
DATASET = RUN_DIR / 'training_smoke.csv'
with DATASET.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(selected)
scratch_config = TrainingConfig(
    dataset_path=str(DATASET), output_dir=str(RUN_DIR / 'scratch_cli'), mode='scratch',
    depth=2, epochs=1, batch_size=2, d_model=16, nhead=2,
    dim_feedforward=32, max_seq_len=128, dropout=0., device='cpu', log_every_steps=1)
SCRATCH_CONFIG = RUN_DIR / 'scratch_config.json'
SCRATCH_CONFIG.write_text(json.dumps(asdict(scratch_config), indent=2), encoding='utf-8')

!simpro-vn-train --config "{SCRATCH_CONFIG}"; echo $? > "{RUN_DIR}/scratch_cli.exitcode"
shell_finished('scratch_cli')
scratch_result = json.loads((Path(scratch_config.output_dir) / 'result.json').read_text())
SCRATCH_CHECKPOINT = Path(scratch_result['best_checkpoint'])
!simpro-vn --checkpoint "{SCRATCH_CHECKPOINT}" --depth 2 --device cpu --text CCO --output "{RUN_DIR}/scratch_reloaded.json"; echo $? > "{RUN_DIR}/scratch_reload.exitcode"
shell_finished('scratch_reload')
assert len(json.loads((RUN_DIR / 'scratch_reloaded.json').read_text())[0]) == 17


In [ ]:
def preserved():
    current = {str(p.resolve()): sha256(p) for p in WEIGHTS_DIR.rglob('*.pt')}
    assert current == original_weights, 'Original weight directory changed'
    return {'unchanged_checkpoints': len(current)}

def finetune_and_resume():
    configuration = TrainingConfig(
        dataset_path=str(DATASET), output_dir=str(RUN_DIR / 'finetune_api'),
        mode='finetune', checkpoint_path=str(SOURCE_CHECKPOINT), depth=6,
        epochs=2, batch_size=2, device=DEVICE, precision='fp32', log_every_steps=1)
    try:
        partial = run_training(configuration, max_epochs=1)
        resumed = run_training(replace(
            configuration, checkpoint_path=None, resume_from=partial['last_checkpoint'],
            output_dir=str(RUN_DIR / 'resume_api')))
        assert partial['completed_epochs'] == 1 and resumed['completed_epochs'] == 2
        assert len(resumed['history']) == 2
        for result in (partial, resumed):
            for key in ('best_checkpoint', 'last_checkpoint'):
                checkpoint = Path(result[key])
                assert checkpoint.is_file() and RUN_DIR.resolve() in checkpoint.resolve().parents
                predictor = ValueNetworkPredictor(checkpoint_path=checkpoint, depth=6, device='cpu')
                assert predictor.describe()['verification'] == 'metadata'
                assert len(predictor.predict('CCO')[0]) == 17
            assert (Path(result['directory']) / 'history.json').is_file()
            from_config = ValueNetworkPredictor.from_config(Path(result['directory']) / 'inference_config.json')
            assert len(from_config.predict('CCO')[0]) == 17
        payload = {'finetune': partial, 'resumed': resumed}
        (RUN_DIR / 'finetuning_results.json').write_text(json.dumps(payload, indent=2))
        return {'completed_epochs': resumed['completed_epochs'], 'result': str(RUN_DIR / 'finetuning_results.json')}
    finally:
        preserved()

phase('finetune_and_resume', finetune_and_resume)
phase('original_weight_preservation', preserved)


## Summary and Export

Every required phase must run. PASS_WITH_SKIPS means executable checks passed while listed hardware capabilities remain unverified. On failure, inspect the cell, summary.json, functional-test log, and matrix reports before changing tolerances. Rerun in a fresh runtime/new RUN_DIR after fixes.

Colab temporary storage disappears when the runtime ends. Download reports.zip to preserve reports/configurations/logs and small data; no .pt files are included. Copy new training subdirectories from RUN_DIR separately to your own persistent location if needed.


In [ ]:
required = {'install', 'cli_help', 'training_help', 'module_help', 'inventory', 'wheel',
            'functional_tests', 'direct_complete', 'direct_prefix', 'direct_trajectory',
            'direct_batch', 'published_cli', 'published_api', 'integration_api',
            'data_preparation', 'scratch_cli', 'scratch_reload', 'finetune_and_resume',
            'original_weight_preservation'}
executed = {p['phase'] for p in phases}
missing = sorted(required - executed)
unexpected = sorted(executed - required)
failed = [p for p in phases if p['status'] != 'PASS']
print('Missing phases:', missing)
print('Unexpected phases:', unexpected)
print('Non-passing phases:', [p['phase'] for p in failed])
assert not missing and not unexpected, f'Incomplete phases: missing={missing}, unexpected={unexpected}'
assert not failed, f'Non-passing phases:{[p["phase"] for p in failed]}'
matrix_report = json.loads((MATRIX_DIR / 'report.json').read_text())
assert len(matrix_report['records']) == 542 and all(row['status'] == 'PASS' for row in matrix_report['records'])
skips = list(test_report['skipped'])
if integration_report['cpu_cuda_parity'].startswith('SKIP'):
    skips.append({'test': 'published_cpu_cuda_parity', 'reason': integration_report['cpu_cuda_parity']})
final_status = 'PASS_WITH_SKIPS' if skips else 'PASS'
report = {'project': 'SimPro-ValueNetwork', 'status': final_status,
          'environment': environment, 'phases': phases, 'skipped_capabilities': skips,
          'published_models': 60, 'published_inference_checks': 542,
          'functional_tests_run': test_report['tests_run'], 'original_weights_unchanged': True,
          'scope': 'Functional validation and consistency; not scientific accuracy or full training'}
(RUN_DIR / 'summary.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))

import zipfile
archive = RUN_DIR / 'reports.zip'
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in RUN_DIR.rglob('*'):
        if path.is_file() and path != archive and path.suffix.lower() not in ('.pt', '.pth', '.tmp'):
            bundle.write(path, path.relative_to(RUN_DIR))
print('Result:', final_status)
print('Report archive:', archive)
# Optional interactive download:
# from google.colab import files
# files.download(str(archive))
